#### Importing Python Libraries

In [1]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import joblib

#### Download NLTK

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\samsa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samsa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\samsa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\samsa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#### Loading the Dataset

In [3]:
df = pd.read_csv("../../Dataset/Tweet Dataset.csv",
                 encoding="ISO-8859-1",
                 names=["Target","ID","Date","Flag","User","Text"])
df.head()

,Target,ID,Date,Flag,User,Text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


#### Replacing the Values in Target Column

In [4]:
df['Target'] = df['Target'].replace(4,1).astype(int)
df.head()

,Target,ID,Date,Flag,User,Text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


#### Feature Selection

In [5]:
df = df[['Text','Target']]
df.head()

,Text,Target
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


In [6]:
print(f"Shape of the Dataset : {df.shape}")
print(f"Number of the Records of the Dataset : {df.shape[0]}")
print(f"Number of the Features of the Dataset : {df.shape[1]}")

Shape of the Dataset : (1600000, 2)
Number of the Records of the Dataset : 1600000
Number of the Features of the Dataset : 2


#### Cleaning The Text

In [7]:
NEGATION_WORDS = {"not", "no", "nor", "never", "n't"}
stop_words = set(stopwords.words('english')).union(ENGLISH_STOP_WORDS)
lemmatizer = WordNetLemmatizer()

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)

    cleaned_tokens = []
    negate = False
    for token in tokens:
        if token in NEGATION_WORDS:
            negate = True
            cleaned_tokens.append(token)
            continue
        if token in stop_words or len(token) <= 2:
            continue
        lemma = lemmatizer.lemmatize(token)
        if negate:
            lemma = "NOT_" + lemma
        cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)

In [9]:
df['Clean_Text'] = df['Text'].apply(clean_text)
df.head()

,Text,Target,Clean_Text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,thats bummer shoulda got david carr day
1,is upset that he can't update his Facebook by ...,0,upset update facebook texting result school to...
2,@Kenichan I dived many times for the ball. Man...,0,dived time ball managed save rest bound
3,my whole body feels itchy and like its on fire,0,body feel itchy like
4,"@nationwideclass no, it's not behaving at all....",0,no not NOT_behaving NOT_mad


#### Final Dataset for Modelling

In [10]:
x = df['Clean_Text']
y = df['Target']

In [11]:
x.head()

0              thats bummer shoulda got david carr day
1    upset update facebook texting result school to...
2              dived time ball managed save rest bound
3                                 body feel itchy like
4                          no not NOT_behaving NOT_mad
Name: Clean_Text, dtype: object

In [12]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Target, dtype: int64

#### Splitting the Dataset into Training and Testing Sets

In [13]:
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [14]:
print("Shape of the Training Sets :")
print(x_train.shape,y_train.shape)

Shape of the Training Sets :
(1280000,) (1280000,)


In [15]:
print("Shape of the Testing Sets :")
print(x_test.shape,y_test.shape)

Shape of the Testing Sets :
(320000,) (320000,)


In [16]:
print("\nTarget distribution in full dataset:")
print(y.value_counts(normalize=True))


Target distribution in full dataset:
Target
0    0.5
1    0.5
Name: proportion, dtype: float64


In [17]:
print("\nTarget distribution in training set:")
print(y_train.value_counts(normalize=True))


Target distribution in training set:
Target
1    0.5
0    0.5
Name: proportion, dtype: float64


In [18]:
print("\nTarget distribution in testing set:")
print(y_test.value_counts(normalize=True))


Target distribution in testing set:
Target
0    0.5
1    0.5
Name: proportion, dtype: float64


#### Creation of the Pipeline for Model Training

In [19]:
model = Pipeline(steps=[
    ("count", CountVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        binary=True,
        max_features=100000
    )),
    ("svm", LinearSVC(
        C=1.0,
        class_weight=None,
        loss="squared_hinge"
    ))
])


model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('count', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [20]:
y_train_pred = model.predict(x_train)
training_accuracy = accuracy_score(y_train,y_train_pred)

In [21]:
y_test_pred = model.predict(x_test)
testing_accuracy = accuracy_score(y_test,y_test_pred)

In [22]:
print(f"Training Accuracy : {training_accuracy}")
print(f"Testing Accuracy : {testing_accuracy}")

Training Accuracy : 0.82372265625
Testing Accuracy : 0.784334375


In [23]:
fit_eval_df = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Accuracy": [
        accuracy_score(y_train, y_train_pred),
        accuracy_score(y_test, y_test_pred)
    ],
    "Precision": [
        precision_score(y_train, y_train_pred),
        precision_score(y_test, y_test_pred)
    ],
    "Recall": [
        recall_score(y_train, y_train_pred),
        recall_score(y_test, y_test_pred)
    ],
    "F1 Score": [
        f1_score(y_train, y_train_pred),
        f1_score(y_test, y_test_pred)
    ]
})

fit_eval_df

,Dataset,Accuracy,Precision,Recall,F1 Score
0,Training,0.823723,0.807709,0.849744,0.828193
1,Testing,0.784334,0.768949,0.812937,0.790332


#### Saving the Model

In [24]:
joblib.dump(model,"../../Models/svc-count.joblib")

['../../Models/svc-count.joblib']